# English to Twi: open models on two native-speaker test sets

Runs NLLB-200 (distilled 600M, distilled 1.3B, 1.3B, 3.3B; target codes `twi_Latn` and `aka_Latn`) and MADLAD-400-3B on the two test sets in `data/twi/`, writes one file per system to `outputs/twi/`, then scores everything with `scripts/score.py`.

**Run:** Runtime > Change runtime type > **A100 GPU** (or T4 with High-RAM) > Save, then Runtime > Run all. The published outputs were made on an A100 in November 2026 with the settings below (fp16 for NLLB, bf16 for MADLAD, beam 4, 128 new tokens). Re-running regenerates them; small differences across GPUs and library versions are possible.

Claude's outputs come from `scripts/run_claude.ts`, which needs an Anthropic API key.

In [ ]:
!pip -q install sacrebleu==2.4.3 sentencepiece
!git clone -q https://github.com/rootabytes/ghanaian-language-eval.git
%cd ghanaian-language-eval
import torch, json, os, time, gc, glob
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > A100 (or T4 + High-RAM)"
GPU = torch.cuda.get_device_name(0)
BATCH = 64 if any(g in GPU for g in ("A100", "H100", "L4")) else 16
print(GPU, "| batch", BATCH)

In [ ]:
SETS = {}
for path in sorted(glob.glob("data/twi/*.jsonl")):
    name = os.path.basename(path)[:-6]
    SETS[name] = [json.loads(l) for l in open(path, encoding="utf-8")]
print({k: len(v) for k, v in SETS.items()})

def out_path(model_id, target):
    return f"outputs/twi/{model_id.split('/')[-1]}__{target}.jsonl"

def done(model_id, target, set_name):
    p = out_path(model_id, target)
    return os.path.exists(p) and any(json.loads(l)["set"] == set_name for l in open(p, encoding="utf-8"))

def save(model_id, target, set_name, rows, cands):
    # Replaces this system's rows for this set, keeps its rows for the other set.
    p = out_path(model_id, target)
    keep = [json.loads(l) for l in open(p, encoding="utf-8")] if os.path.exists(p) else []
    keep = [r for r in keep if r["set"] != set_name]
    keep += [dict(set=set_name, id=r["id"], model=model_id, target=target, candidate=c) for r, c in zip(rows, cands)]
    keep.sort(key=lambda r: (r["set"], str(r["id"])))
    with open(p, "w", encoding="utf-8") as f:
        for r in keep: f.write(json.dumps(r, ensure_ascii=False) + "
")
    print(f"  {model_id:34s} {target:9s} {set_name:10s} saved", flush=True)

# Set FORCE = True to regenerate the published outputs instead of skipping them.
FORCE = False
def done_keys():
    return set() if FORCE else {(m, t, s) for m in NLLB + ["google/madlad400-3b-mt"]
                                for t in ("twi_Latn", "aka_Latn", "ak") for s in SETS if done(m, t, s)}
NLLB = ["facebook/nllb-200-distilled-600M", "facebook/nllb-200-distilled-1.3B",
        "facebook/nllb-200-1.3B", "facebook/nllb-200-3.3B"]

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
NLLB = ["facebook/nllb-200-distilled-600M", "facebook/nllb-200-distilled-1.3B",
        "facebook/nllb-200-1.3B", "facebook/nllb-200-3.3B"]
TARGETS = ["twi_Latn", "aka_Latn"]
for mid in NLLB:
    todo = [(t, s) for t in TARGETS for s in SETS if (mid, t, s) not in done_keys()]
    if not todo: continue
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(mid, src_lang="eng_Latn")
    model = AutoModelForSeq2SeqLM.from_pretrained(mid, dtype=torch.float16, low_cpu_mem_usage=True, device_map="cuda").eval()
    model.generation_config.max_length = None
    for tgt, set_name in todo:
        rows, cands, bos = SETS[set_name], [], tok.convert_tokens_to_ids(tgt)
        for i in range(0, len(rows), BATCH):
            enc = tok([r["english"] for r in rows[i:i+BATCH]], return_tensors="pt", padding=True).to("cuda")
            with torch.no_grad():
                out = model.generate(**enc, forced_bos_token_id=bos, num_beams=4, max_new_tokens=128)
            cands += tok.batch_decode(out, skip_special_tokens=True)
        save(mid, tgt, set_name, rows, cands)
    del model; gc.collect(); torch.cuda.empty_cache()
    print(f"   ({(time.time()-t0)/60:.1f} min)")

In [ ]:
# MADLAD-400-3B. T5-family models overflow in fp16, so bf16 where the GPU has it, else fp32.
# Same decoding as the int8/CPU run in translate_madlad.py: beam 4, 128 tokens, repetition penalty 1.1,
# no repeated 4-grams, and the "<2ak> " prefix encoded together with the sentence.
from transformers import T5ForConditionalGeneration, T5Tokenizer
mid = "google/madlad400-3b-mt"
todo = [s for s in SETS if (mid, "ak", s) not in done_keys()]
if todo:
    t0 = time.time()
    dt = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
    tok = T5Tokenizer.from_pretrained(mid)
    model = T5ForConditionalGeneration.from_pretrained(mid, dtype=dt, low_cpu_mem_usage=True, device_map="cuda").eval()
    for set_name in todo:
        rows, cands = SETS[set_name], []
        for i in range(0, len(rows), BATCH):
            enc = tok(["<2ak> " + r["english"] for r in rows[i:i+BATCH]], return_tensors="pt", padding=True).to("cuda")
            with torch.no_grad():
                out = model.generate(**enc, num_beams=4, max_new_tokens=128, repetition_penalty=1.1, no_repeat_ngram_size=4)
            cands += tok.batch_decode(out, skip_special_tokens=True)
        save(mid, "ak", set_name, rows, cands)
    del model; gc.collect(); torch.cuda.empty_cache()
    print(f"   ({(time.time()-t0)/60:.1f} min)")

In [ ]:
!python -X utf8 scripts/score.py twi
!python -X utf8 scripts/bootstrap.py twi crowd674 "claude-sonnet-5|prompt:Akuapem" "facebook/nllb-200-3.3B|aka_Latn"